In [1]:
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


### Load preprocessed dataset

In [2]:
import json

npzfile = np.load('kor-eng.npz')
x_train = npzfile['x_train']
y_train = npzfile['y_train']

with open("kor-eng-krvocab.json", "rb") as f:
    x_vocab = json.load(f)
with open("kor-eng-envocab.json", "rb") as f:
    y_vocab = json.load(f)
    
inverse_x_vocab = {index: token for token, index in x_vocab.items()}
inverse_y_vocab = {index: token for token, index in y_vocab.items()}

### Building models

In [3]:
BUFFER_SIZE = len(x_train)
BATCH_SIZE = 16
embedding_dim = 1024
latent_dim = 1024
x_vocab_size = len(x_vocab)
y_vocab_size = len(y_vocab)

class TranslationDataset(Dataset):
    def __init__(self, x_data, y_data):
        self.x_data = torch.LongTensor(x_data)
        self.y_data = torch.LongTensor(y_data)

    def __len__(self):
        return len(self.x_data)

    def __getitem__(self, idx):
        return self.x_data[idx], self.y_data[idx]

dataset = TranslationDataset(x_train, y_train)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

In [33]:
for example_input_batch, example_target_batch in dataloader:
    break

In [55]:
example_input_batch.shape, example_target_batch.shape

(torch.Size([16, 166]), torch.Size([16, 113]))

In [57]:
# Compact batch (remove trailing zeros)
def compact_batch(batch):
    max_len = (batch != 0).sum(dim=1).max().item()
    return batch[:, :max_len]

example_input_batch = compact_batch(example_input_batch)
example_target_batch = compact_batch(example_target_batch)

In [58]:
example_input_batch.shape, example_target_batch.shape

(torch.Size([16, 22]), torch.Size([16, 14]))

### Define and test encoder

In [6]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers=3):
        super(Encoder, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.gru = nn.GRU(embedding_dim, hidden_dim, num_layers=num_layers, batch_first=True)

    def forward(self, x):
        # x: (batch, seq_len)
        lengths = (x != 0).sum(dim=1).cpu()

        embedded = self.embedding(x)  # (batch, seq_len, embedding_dim)

        # pack sequence for efficient computation, skipping padded positions
        packed = nn.utils.rnn.pack_padded_sequence(
            embedded, lengths, batch_first=True, enforce_sorted=False
        )

        packed_out, hidden = self.gru(packed)

        output, _ = nn.utils.rnn.pad_packed_sequence(packed_out, batch_first=True)

        # hidden: (num_layers, batch, hidden_dim)
        s1 = hidden[0]
        s2 = hidden[1]
        s3 = hidden[2]

        return output, s1, s2, s3

In [59]:
encoder = Encoder(x_vocab_size, embedding_dim, latent_dim).to(device)
print(encoder)

Encoder(
  (embedding): Embedding(2552, 1024, padding_idx=0)
  (gru): GRU(1024, 1024, num_layers=3, batch_first=True)
)


In [60]:
last_output, last_state1, last_state2, last_state3 = \
    encoder(example_input_batch.to(device))
print(last_output.shape, last_state1.shape)

torch.Size([16, 22, 1024]) torch.Size([16, 1024])


### Define and test decoder

In [9]:
class Decoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers=3):
        super(Decoder, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.gru = nn.GRU(embedding_dim, hidden_dim, num_layers=num_layers, batch_first=True)

    def forward(self, x, s1, s2, s3):
        # x: (batch, seq_len)
        # s1, s2, s3: (batch, hidden_dim), one per GRU layer
        lengths = (x != 0).sum(dim=1).cpu()
        lengths = lengths.clamp(min=1)

        embedded = self.embedding(x)  # (batch, seq_len, embedding_dim)

        packed = nn.utils.rnn.pack_padded_sequence(
            embedded, lengths, batch_first=True, enforce_sorted=False
        )

        hidden = torch.stack([s1, s2, s3], dim=0)  # (num_layers, batch, hidden_dim)

        packed_out, out_hidden = self.gru(packed, hidden)

        output, _ = nn.utils.rnn.pad_packed_sequence(packed_out, batch_first=True)

        out_s1 = out_hidden[0]
        out_s2 = out_hidden[1]
        out_s3 = out_hidden[2]

        # note: unlike the no-attention decoder, this returns the raw GRU
        # output sequence instead of vocab logits -- the attention layer
        # combines it with the encoder's output before projecting to vocab
        return output, out_s1, out_s2, out_s3

In [61]:
decoder = Decoder(y_vocab_size, embedding_dim, latent_dim).to(device)

In [62]:
dec_output, s1, s2, s3 = decoder(
    example_target_batch.to(device), last_state1, last_state2, last_state3)

In [63]:
print(dec_output.shape, s1.shape, s2.shape, s3.shape)

torch.Size([16, 14, 1024]) torch.Size([16, 1024]) torch.Size([16, 1024]) torch.Size([16, 1024])


### Define and test attention layer

In [ ]:
class Attention(nn.Module):
    def __init__(self, hidden_dim, vocab_size):
        super(Attention, self).__init__()
        # query: the output of decoder's RNN
        self.query_proj = nn.Linear(hidden_dim, hidden_dim, bias=False)
        # value: the output of encoder (note: key is computed using value HERE)
        self.key_proj = nn.Linear(hidden_dim, hidden_dim, bias=False)
        # trainable scale vector, analogous to keras' AdditiveAttention
        self.scale = nn.Parameter(torch.ones(hidden_dim))
        self.fc = nn.Linear(hidden_dim * 2, vocab_size)

    def forward(self, dec_output, enc_output, target_mask, input_mask):
        ## --- fill here ---

In [76]:
attention_layer = Attention(latent_dim, y_vocab_size).to(device)

In [77]:
query_mask = (example_target_batch != 0).to(device)
value_mask = (example_input_batch != 0).to(device)

logits, a = attention_layer(dec_output, last_output, query_mask, value_mask)
print(logits.shape, a.shape)

torch.Size([16, 14, 2527]) torch.Size([16, 14, 22])


### Loss function

In [69]:
loss_fn = nn.CrossEntropyLoss(reduction='none')

In [70]:
def batch_loss(y_true, y_pred):
    # y_true: (batch, seq_len)
    # y_pred: (batch, seq_len, vocab_size)
    batch_size, seq_len, vocab_size = y_pred.shape
    loss = loss_fn(y_pred.reshape(-1, vocab_size), y_true.reshape(-1))
    loss = loss.reshape(batch_size, seq_len)

    mask = (y_true != 0).float()
    loss = loss * mask

    return loss.sum() / mask.sum()

In [71]:
batch_loss(example_target_batch[:, 1:].to(device), logits[:, :-1, :])

tensor(7.8286, device='cuda:0', grad_fn=<DivBackward0>)

### Training

In [72]:
optimizer = optim.Adam(
    list(encoder.parameters()) + list(decoder.parameters()) + list(attention_layer.parameters()),
    lr=1e-4
)

In [ ]:
def predict(x_batch, y_batch, training=True):
    encoder.train(training)
    decoder.train(training)
    ## --- fill here ---

In [74]:
def train_step(x_batch, y_batch):
    # 1. compact batch tensors
    x_batch = compact_batch(x_batch).to(device)
    y_batch = compact_batch(y_batch).to(device)

    optimizer.zero_grad()

    logits = predict(x_batch, y_batch, training=True)

    # 3. loss
    loss = batch_loss(y_batch[:, 1:], logits[:, :-1, :])

    loss.backward()
    optimizer.step()

    return loss.item()

In [78]:
for epoch in range(100):
    start = time.time()

    loss_sum = 0
    for x_batch, y_batch in dataloader:
        loss = train_step(x_batch, y_batch)
        loss_sum += loss

    print('Time for epoch {} is {:.2f} sec: training loss = {:.6f}'.format(
        epoch + 1, time.time() - start, loss))

Time for epoch 1 is 10.52 sec: training loss = 4.093348
Time for epoch 2 is 10.29 sec: training loss = 2.666712
Time for epoch 3 is 9.98 sec: training loss = 2.948402
Time for epoch 4 is 10.19 sec: training loss = 4.139038
Time for epoch 5 is 10.47 sec: training loss = 3.070266
Time for epoch 6 is 10.61 sec: training loss = 4.006010
Time for epoch 7 is 10.76 sec: training loss = 4.264124
Time for epoch 8 is 10.67 sec: training loss = 2.156456
Time for epoch 9 is 10.68 sec: training loss = 3.705450
Time for epoch 10 is 10.66 sec: training loss = 2.442017
Time for epoch 11 is 10.74 sec: training loss = 3.099241
Time for epoch 12 is 10.19 sec: training loss = 3.288886
Time for epoch 13 is 10.40 sec: training loss = 3.172289
Time for epoch 14 is 10.27 sec: training loss = 1.743058
Time for epoch 15 is 10.11 sec: training loss = 2.781015
Time for epoch 16 is 10.10 sec: training loss = 3.286700
Time for epoch 17 is 10.49 sec: training loss = 0.913475
Time for epoch 18 is 10.72 sec: training 

In [ ]:
torch.save(encoder.state_dict(), 'nmt-w-attention.encoder.pt')
torch.save(decoder.state_dict(), 'nmt-w-attention.decoder.pt')
torch.save(attention_layer.state_dict(), 'nmt-w-attention.attention.pt')

### Test translation

In [79]:
from kiwipiepy import Kiwi
kiwi = Kiwi()

In [80]:
def lemmatize_korean(text):
    return [t.form for t in kiwi.tokenize(text)]

In [81]:
def beam_translate(src, max_steps=100, k=16):
    encoder.eval()
    decoder.eval()
    attention_layer.eval()

    # tokenization
    src_tokens = np.array([x_vocab['<bos>']] + [x_vocab[x] for x in lemmatize_korean(src)] + [x_vocab['<eos>']])
    #print(lemmatize_korean(src))

    # add the batch axis
    x_test = torch.LongTensor(src_tokens).unsqueeze(0).to(device)

    with torch.no_grad():
        # compute encoder and get hidden status
        enc_output, s1, s2, s3 = encoder(x_test)

        # init candidates
        last_token = torch.LongTensor([[y_vocab['<bos>']]]).to(device)
        candidates = [(0., last_token, s1, s2, s3, [y_vocab['<bos>']], False)]

        for _ in range(max_steps):
            new_candidates = []

            for score, token, c_s1, c_s2, c_s3, output_seq, eos in candidates:
                # if the candidate already ends,
                if eos:
                    new_candidates.append((score, token, c_s1, c_s2, c_s3, output_seq, eos))
                    continue

                # compute the prob. of following tokens
                dec_output, new_s1, new_s2, new_s3 = decoder(token, c_s1, c_s2, c_s3)
                logits, _ = attention_layer(dec_output, enc_output, token != 0, x_test != 0)
                # shape of logits: (1, 1, vocab_size)
                probs = torch.log_softmax(logits, dim=2)

                # use the token with the top-k logits as the input
                # of the decoder at the next time step
                values, indices = torch.topk(probs.squeeze(), k=k)

                for prob, idx in zip(values, indices):
                    idx_val = idx.item()
                    # if prediction is eos, output sequence is complete
                    is_eos = (idx_val == y_vocab['<eos>'])

                    new_token = torch.LongTensor([[idx_val]]).to(device)
                    new_candidates.append(
                        (score + prob.item(), new_token, new_s1, new_s2, new_s3,
                         output_seq + [idx_val], is_eos)
                    )

            candidates = sorted(new_candidates, key=lambda t: -t[0])[:k]

    return [(candidates[i][0], ' '.join([inverse_y_vocab[x] for x in candidates[i][5]])) for i in range(k)]

In [86]:
beam_translate('톰은 프랑스어를 매일 공부한다.', k=32)

[(-5.43944055493921, '<bos> tom has to learn french . <eos>'),
 (-8.397185355424881, '<bos> tom has to study french . <eos>'),
 (-8.674674148671329, '<bos> tom has . <eos>'),
 (-9.139032210223377, '<bos> tom has learn french . <eos>'),
 (-9.450532674789429, '<bos> tom has french learn french . <eos>'),
 (-10.133265013806522, '<bos> tom has nodded french . <eos>'),
 (-10.354750390164554, '<bos> tom has study french . <eos>'),
 (-10.83830043207854, '<bos> tom has to speak french . <eos>'),
 (-11.162186742760241, '<bos> tom has seemed to learn french . <eos>'),
 (-11.45651435572654, '<bos> tom has to learn french tom . <eos>'),
 (-11.799130089581013, '<bos> tom has to learn french french . <eos>'),
 (-12.67797205876559, '<bos> tom has learn french to tom . <eos>'),
 (-12.710531665943563, '<bos> tom has learn french tom . <eos>'),
 (-12.925827488303185, '<bos> tom has to learn french mary . <eos>'),
 (-13.035964991897345, '<bos> tom has learned a few words in french . <eos>'),
 (-13.111461